# Model Comparison

This notebook compares the results produced by the six baseline experiments and the advanced DAVIS experiment.

The notebook does **not train any model**. It reads the saved metrics files from `results/metrics/` and produces consistent comparison tables and figures.

The comparisons are kept scientifically separate:

- DAVIS baseline models: GCN + CNN, GAT + CNN, and GIN + CNN
- KIBA baseline models: GCN + CNN, GAT + CNN, and GIN + CNN
- DAVIS baselines versus the advanced GINE + frozen ESM-2 model

DAVIS and KIBA RMSE values are **not directly compared with each other** because the two datasets use different affinity target scales.

## Notebook workflow

```text
Saved metrics from model notebooks
        │
        ├── DAVIS GCN + CNN
        ├── DAVIS GAT + CNN
        ├── DAVIS GIN + CNN
        ├── KIBA GCN + CNN
        ├── KIBA GAT + CNN
        ├── KIBA GIN + CNN
        └── DAVIS Advanced GINE + ESM-2
        │
        ▼
Validate result files and columns
        │
        ▼
Combine all metrics into one table
        │
        ├─────────────────────────────┐
        ▼                             ▼
DAVIS comparisons                KIBA comparisons
        │                             │
        ▼                             ▼
Random / Cold-drug /            Random / Cold-drug /
Cold-target                     Cold-target
        │                             │
        └──────────────┬──────────────┘
                       ▼
              Rank models per split
                       │
                       ▼
         Compare DAVIS advanced model
             with DAVIS baselines
                       │
                       ▼
          Save comparison CSV files
              and summary figures
```

Lower MSE, RMSE, and MAE indicate better regression performance.  
Higher Concordance Index and Pearson correlation indicate better ranking and agreement.

# Part A — Setup

## 1. Imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

## 2. Locate the project folder

In [ ]:
def find_project_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/final_aml_gnn_dta"),
    ]

    content_dir = Path("/content")

    if content_dir.exists():
        candidates.extend(
            path.parent
            for path in content_dir.glob("*/config.py")
        )

    for candidate in candidates:
        if (
            (candidate / "config.py").exists()
            and (candidate / "results").exists()
        ):
            return candidate.resolve()

    raise FileNotFoundError(
        "Project root could not be found. "
        "Run this notebook from the final_aml_gnn_dta project."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 3. Project result folders

In [ ]:
from config import (
    FIGURES_DIR,
    METRICS_DIR,
)

COMPARISON_DIR = METRICS_DIR / "comparisons"
COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Metrics folder:", METRICS_DIR)
print("Figures folder:", FIGURES_DIR)
print("Comparison folder:", COMPARISON_DIR)

# Part B — Load and validate model results

## 4. Expected metrics files

Each model notebook should save one CSV file containing three rows: random, cold-drug, and cold-target.

The advanced DAVIS notebook follows the same format.

In [ ]:
RESULT_FILES = {
    "davis_gcn": METRICS_DIR / "davis_gcn_cnn.csv",
    "davis_gat": METRICS_DIR / "davis_gat_cnn.csv",
    "davis_gin": METRICS_DIR / "davis_gin_cnn.csv",
    "kiba_gcn": METRICS_DIR / "kiba_gcn_cnn.csv",
    "kiba_gat": METRICS_DIR / "kiba_gat_cnn.csv",
    "kiba_gin": METRICS_DIR / "kiba_gin_cnn.csv",
    "davis_advanced": (
        METRICS_DIR
        / "davis_advanced_gine_esm2.csv"
    ),
}

file_check = pd.DataFrame(
    {
        "result": RESULT_FILES.keys(),
        "file": [
            path.name
            for path in RESULT_FILES.values()
        ],
        "exists": [
            path.exists()
            for path in RESULT_FILES.values()
        ],
    }
)

display(file_check)

missing_files = [
    path.name
    for path in RESULT_FILES.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following result files are missing. "
        "Run the corresponding model notebooks first: "
        + ", ".join(missing_files)
    )

## 5. Load and check the result tables

In [ ]:
REQUIRED_COLUMNS = {
    "dataset",
    "model",
    "split",
    "best_epoch",
    "completed_epochs",
    "best_validation_mse",
    "test_mse",
    "test_rmse",
    "test_mae",
    "ci",
    "pearson",
    "trainable_parameters",
}


def load_result_file(file_path: Path) -> pd.DataFrame:
    result_df = pd.read_csv(file_path)

    missing_columns = (
        REQUIRED_COLUMNS
        .difference(result_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    expected_splits = {
        "random",
        "cold_drug",
        "cold_target",
    }

    found_splits = set(
        result_df["split"].astype(str)
    )

    if found_splits != expected_splits:
        raise ValueError(
            f"{file_path.name} must contain exactly "
            f"{sorted(expected_splits)}. "
            f"Found {sorted(found_splits)}."
        )

    return result_df.copy()


result_tables = [
    load_result_file(file_path)
    for file_path in RESULT_FILES.values()
]

all_results_df = pd.concat(
    result_tables,
    ignore_index=True,
)

print(
    "Combined result rows:",
    len(all_results_df),
)

display(
    all_results_df.head()
)

## 6. Save the combined result table

In [ ]:
all_results_path = (
    COMPARISON_DIR
    / "all_model_results.csv"
)

all_results_df.to_csv(
    all_results_path,
    index=False,
)

print(
    "Combined results saved to:",
    all_results_path,
)

# Part C — DAVIS baseline comparison

## 7. DAVIS baseline results

This section compares only the three baseline drug encoders while keeping the protein CNN and prediction head consistent.

In [ ]:
davis_baseline_df = all_results_df[
    (all_results_df["dataset"] == "DAVIS")
    & (
        all_results_df["model"].isin(
            [
                "GCN + CNN",
                "GAT + CNN",
                "GIN + CNN",
            ]
        )
    )
].copy()

davis_baseline_df = (
    davis_baseline_df
    .sort_values(
        ["split", "test_rmse"]
    )
    .reset_index(drop=True)
)

display(
    davis_baseline_df[
        [
            "model",
            "split",
            "test_rmse",
            "test_mae",
            "ci",
            "pearson",
            "best_epoch",
        ]
    ].round(4)
)

## 8. Rank DAVIS baseline models within each split

In [ ]:
davis_baseline_ranked = (
    davis_baseline_df.copy()
)

davis_baseline_ranked["rmse_rank"] = (
    davis_baseline_ranked
    .groupby("split")["test_rmse"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

davis_baseline_ranked = (
    davis_baseline_ranked
    .sort_values(
        ["split", "rmse_rank"]
    )
    .reset_index(drop=True)
)

display(
    davis_baseline_ranked[
        [
            "split",
            "rmse_rank",
            "model",
            "test_rmse",
            "ci",
            "pearson",
        ]
    ].round(4)
)

davis_baseline_path = (
    COMPARISON_DIR
    / "davis_baseline_comparison.csv"
)

davis_baseline_ranked.to_csv(
    davis_baseline_path,
    index=False,
)

print("Saved:", davis_baseline_path)

## 9. DAVIS baseline RMSE figure

In [ ]:
split_order = [
    "random",
    "cold_drug",
    "cold_target",
]

model_order = [
    "GCN + CNN",
    "GAT + CNN",
    "GIN + CNN",
]

x = np.arange(
    len(split_order)
)

width = 0.24

plt.figure(figsize=(9, 5))

for index, model_name in enumerate(model_order):
    model_values = (
        davis_baseline_df[
            davis_baseline_df["model"]
            == model_name
        ]
        .set_index("split")
        .reindex(split_order)
        ["test_rmse"]
    )

    plt.bar(
        x + (index - 1) * width,
        model_values,
        width=width,
        label=model_name,
    )

plt.xticks(
    x,
    [
        "Random",
        "Cold-drug",
        "Cold-target",
    ],
)
plt.ylabel("Test RMSE")
plt.xlabel("Data split")
plt.title(
    "DAVIS Baseline Model Comparison"
)
plt.legend()
plt.tight_layout()

davis_baseline_figure = (
    FIGURES_DIR
    / "davis_baseline_rmse_comparison.png"
)

plt.savefig(
    davis_baseline_figure,
    dpi=200,
    bbox_inches="tight",
)
plt.show()

print("Saved:", davis_baseline_figure)

# Part D — KIBA baseline comparison

## 10. KIBA baseline results

KIBA is analysed separately because its affinity score is on a different scale from DAVIS pKd.

In [ ]:
kiba_baseline_df = all_results_df[
    all_results_df["dataset"] == "KIBA"
].copy()

kiba_baseline_df = (
    kiba_baseline_df
    .sort_values(
        ["split", "test_rmse"]
    )
    .reset_index(drop=True)
)

display(
    kiba_baseline_df[
        [
            "model",
            "split",
            "test_rmse",
            "test_mae",
            "ci",
            "pearson",
            "best_epoch",
        ]
    ].round(4)
)

## 11. Rank KIBA baseline models within each split

In [ ]:
kiba_baseline_ranked = (
    kiba_baseline_df.copy()
)

kiba_baseline_ranked["rmse_rank"] = (
    kiba_baseline_ranked
    .groupby("split")["test_rmse"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

kiba_baseline_ranked = (
    kiba_baseline_ranked
    .sort_values(
        ["split", "rmse_rank"]
    )
    .reset_index(drop=True)
)

display(
    kiba_baseline_ranked[
        [
            "split",
            "rmse_rank",
            "model",
            "test_rmse",
            "ci",
            "pearson",
        ]
    ].round(4)
)

kiba_baseline_path = (
    COMPARISON_DIR
    / "kiba_baseline_comparison.csv"
)

kiba_baseline_ranked.to_csv(
    kiba_baseline_path,
    index=False,
)

print("Saved:", kiba_baseline_path)

## 12. KIBA baseline RMSE figure

In [ ]:
plt.figure(figsize=(9, 5))

for index, model_name in enumerate(model_order):
    model_values = (
        kiba_baseline_df[
            kiba_baseline_df["model"]
            == model_name
        ]
        .set_index("split")
        .reindex(split_order)
        ["test_rmse"]
    )

    plt.bar(
        x + (index - 1) * width,
        model_values,
        width=width,
        label=model_name,
    )

plt.xticks(
    x,
    [
        "Random",
        "Cold-drug",
        "Cold-target",
    ],
)
plt.ylabel("Test RMSE")
plt.xlabel("Data split")
plt.title(
    "KIBA Baseline Model Comparison"
)
plt.legend()
plt.tight_layout()

kiba_baseline_figure = (
    FIGURES_DIR
    / "kiba_baseline_rmse_comparison.png"
)

plt.savefig(
    kiba_baseline_figure,
    dpi=200,
    bbox_inches="tight",
)
plt.show()

print("Saved:", kiba_baseline_figure)

# Part E — DAVIS baseline versus advanced model

## 13. Combine all DAVIS models

The advanced architecture is compared only with the DAVIS baseline models because it was trained and evaluated on DAVIS using the same split definitions.

In [ ]:
davis_all_models_df = all_results_df[
    all_results_df["dataset"] == "DAVIS"
].copy()

davis_all_models_df = (
    davis_all_models_df
    .sort_values(
        ["split", "test_rmse"]
    )
    .reset_index(drop=True)
)

display(
    davis_all_models_df[
        [
            "model",
            "split",
            "test_rmse",
            "test_mae",
            "ci",
            "pearson",
            "trainable_parameters",
        ]
    ].round(4)
)

## 14. Rank all DAVIS models by split

In [ ]:
davis_all_ranked = (
    davis_all_models_df.copy()
)

davis_all_ranked["rmse_rank"] = (
    davis_all_ranked
    .groupby("split")["test_rmse"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

davis_all_ranked = (
    davis_all_ranked
    .sort_values(
        ["split", "rmse_rank"]
    )
    .reset_index(drop=True)
)

display(
    davis_all_ranked[
        [
            "split",
            "rmse_rank",
            "model",
            "test_rmse",
            "test_mae",
            "ci",
            "pearson",
        ]
    ].round(4)
)

davis_all_path = (
    COMPARISON_DIR
    / "davis_all_models_comparison.csv"
)

davis_all_ranked.to_csv(
    davis_all_path,
    index=False,
)

print("Saved:", davis_all_path)

## 15. DAVIS RMSE comparison including the advanced model

In [ ]:
davis_model_order = (
    davis_all_models_df["model"]
    .drop_duplicates()
    .tolist()
)

x_advanced = np.arange(
    len(split_order)
)

advanced_width = (
    0.8 / len(davis_model_order)
)

plt.figure(figsize=(11, 5))

for index, model_name in enumerate(
    davis_model_order
):
    model_values = (
        davis_all_models_df[
            davis_all_models_df["model"]
            == model_name
        ]
        .set_index("split")
        .reindex(split_order)
        ["test_rmse"]
    )

    offset = (
        index
        - (len(davis_model_order) - 1) / 2
    ) * advanced_width

    plt.bar(
        x_advanced + offset,
        model_values,
        width=advanced_width,
        label=model_name,
    )

plt.xticks(
    x_advanced,
    [
        "Random",
        "Cold-drug",
        "Cold-target",
    ],
)
plt.ylabel("Test RMSE")
plt.xlabel("Data split")
plt.title(
    "DAVIS Baseline and Advanced Model Comparison"
)
plt.legend(
    fontsize=8,
)
plt.tight_layout()

davis_all_figure = (
    FIGURES_DIR
    / "davis_all_models_rmse_comparison.png"
)

plt.savefig(
    davis_all_figure,
    dpi=200,
    bbox_inches="tight",
)
plt.show()

print("Saved:", davis_all_figure)

## 16. Compare Concordance Index on DAVIS

CI provides a complementary view to RMSE by measuring whether the model ranks stronger and weaker interactions in the correct order.

In [ ]:
plt.figure(figsize=(11, 5))

for index, model_name in enumerate(
    davis_model_order
):
    model_values = (
        davis_all_models_df[
            davis_all_models_df["model"]
            == model_name
        ]
        .set_index("split")
        .reindex(split_order)
        ["ci"]
    )

    offset = (
        index
        - (len(davis_model_order) - 1) / 2
    ) * advanced_width

    plt.bar(
        x_advanced + offset,
        model_values,
        width=advanced_width,
        label=model_name,
    )

plt.xticks(
    x_advanced,
    [
        "Random",
        "Cold-drug",
        "Cold-target",
    ],
)
plt.ylabel("Concordance Index")
plt.xlabel("Data split")
plt.title(
    "DAVIS Concordance Index by Model"
)
plt.legend(
    fontsize=8,
)
plt.tight_layout()

davis_ci_figure = (
    FIGURES_DIR
    / "davis_all_models_ci_comparison.png"
)

plt.savefig(
    davis_ci_figure,
    dpi=200,
    bbox_inches="tight",
)
plt.show()

print("Saved:", davis_ci_figure)

# Part F — Best models and generalisation summary

## 17. Best model for each dataset and split

In [ ]:
best_models_df = (
    all_results_df
    .sort_values("test_rmse")
    .groupby(
        ["dataset", "split"],
        as_index=False,
    )
    .first()
)

best_models_df = (
    best_models_df[
        [
            "dataset",
            "split",
            "model",
            "test_rmse",
            "test_mae",
            "ci",
            "pearson",
        ]
    ]
    .sort_values(
        ["dataset", "split"]
    )
    .reset_index(drop=True)
)

display(
    best_models_df.round(4)
)

best_models_path = (
    COMPARISON_DIR
    / "best_model_by_dataset_and_split.csv"
)

best_models_df.to_csv(
    best_models_path,
    index=False,
)

print("Saved:", best_models_path)

## 18. Random-to-cold RMSE change

This table shows how much RMSE changes from the random split to each cold-start setting.

A positive value indicates worse performance under the cold-start setting.  
A negative value indicates lower RMSE than the random split for that particular experiment and should be interpreted together with the split composition rather than automatically as easier generalisation.

In [ ]:
rmse_wide = (
    all_results_df
    .pivot_table(
        index=[
            "dataset",
            "model",
        ],
        columns="split",
        values="test_rmse",
    )
    .reset_index()
)

rmse_wide[
    "cold_drug_rmse_change"
] = (
    rmse_wide["cold_drug"]
    - rmse_wide["random"]
)

rmse_wide[
    "cold_target_rmse_change"
] = (
    rmse_wide["cold_target"]
    - rmse_wide["random"]
)

display(
    rmse_wide[
        [
            "dataset",
            "model",
            "random",
            "cold_drug",
            "cold_drug_rmse_change",
            "cold_target",
            "cold_target_rmse_change",
        ]
    ].round(4)
)

generalisation_path = (
    COMPARISON_DIR
    / "cold_start_rmse_change.csv"
)

rmse_wide.to_csv(
    generalisation_path,
    index=False,
)

print("Saved:", generalisation_path)

# Part G — Final comparison summary

## 19. Final result tables

In [ ]:
print("DAVIS baseline ranking")
display(
    davis_baseline_ranked[
        [
            "split",
            "rmse_rank",
            "model",
            "test_rmse",
            "ci",
        ]
    ].round(4)
)

print()
print("KIBA baseline ranking")
display(
    kiba_baseline_ranked[
        [
            "split",
            "rmse_rank",
            "model",
            "test_rmse",
            "ci",
        ]
    ].round(4)
)

print()
print("DAVIS baseline + advanced ranking")
display(
    davis_all_ranked[
        [
            "split",
            "rmse_rank",
            "model",
            "test_rmse",
            "ci",
        ]
    ].round(4)
)

## 20. Automatic summary for dissertation notes

The statements printed below are descriptive summaries of the saved experimental results. They should still be interpreted in the dissertation together with the full metric tables and the characteristics of each split.

In [ ]:
print("Best models by dataset and split")
print("-" * 70)

for _, row in best_models_df.iterrows():
    print(
        f'{row["dataset"]} | {row["split"]}: '
        f'{row["model"]} | '
        f'RMSE={row["test_rmse"]:.4f} | '
        f'CI={row["ci"]:.4f}'
    )

print()
print("DAVIS advanced-model position")
print("-" * 70)

advanced_rows = davis_all_ranked[
    davis_all_ranked["model"].str.contains(
        "GINE",
        case=False,
        na=False,
    )
]

display(
    advanced_rows[
        [
            "split",
            "rmse_rank",
            "test_rmse",
            "test_mae",
            "ci",
            "pearson",
        ]
    ].round(4)
)

## 21. Saved comparison outputs

In [ ]:
saved_outputs = pd.DataFrame(
    {
        "output": [
            "All model results",
            "DAVIS baseline comparison",
            "KIBA baseline comparison",
            "DAVIS all-model comparison",
            "Best model by split",
            "Cold-start RMSE changes",
            "DAVIS baseline RMSE figure",
            "KIBA baseline RMSE figure",
            "DAVIS all-model RMSE figure",
            "DAVIS all-model CI figure",
        ],
        "path": [
            all_results_path,
            davis_baseline_path,
            kiba_baseline_path,
            davis_all_path,
            best_models_path,
            generalisation_path,
            davis_baseline_figure,
            kiba_baseline_figure,
            davis_all_figure,
            davis_ci_figure,
        ],
    }
)

display(saved_outputs)

### Notebook completion

This notebook completes the quantitative comparison stage of the project.

The key methodological rules maintained here are:

- model rankings are calculated **within each dataset and split**;
- DAVIS and KIBA absolute RMSE values are not treated as directly comparable because their target scales differ;
- the advanced architecture is compared directly with the DAVIS baselines under the same random, cold-drug, and cold-target evaluation settings.

The resulting tables and figures can be used in the dissertation Results and Discussion chapters and to select the model used for the downstream AML screening stage.